<a href="https://colab.research.google.com/github/shaktidharreddy/Agentic_AI_eduhubspot_aug_sep_2026/blob/main/ModuleGAI_2/Prompt_Engineering_%26_Context_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Lab 1: Build a prompt engineering workbench using OpenAI API**

**Install Required Libraries**

In [1]:
# Install required libraries in quiet mode to keep the output clean:
# - openai: Official SDK for interacting with OpenAI models
# - pandas: For handling data analysis, tabular storage, and results exporting
!pip install -q openai pandas

**Enter OpenAI API key**

In [3]:
from google.colab import userdata
import os

# Retrieve the OpenAI API key securely from Google Colab's user secrets manager
openai_key = userdata.get("OPENAI_API_KEY")

# Export the loaded key into environment variables so that the OpenAI SDK client can automatically detect it
os.environ["OPENAI_API_KEY"] = openai_key

**Verify**

In [4]:
print("API Key Loaded:", os.environ.get("OPENAI_API_KEY")[:10])

API Key Loaded: sk-proj-lf


**Create OpenAI Client**

In [5]:
from openai import OpenAI
import pandas as pd

# Initialize the official OpenAI API client wrapper using the key set in the environment variables
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Specify the target LLM generation model we want to test/evaluate
MODEL = "gpt-4.1-mini"

print("Client initialized successfully")

Client initialized successfully


**Test API Connection**

In [6]:
# Validate the API client setup by running a simple generation test
test_response = client.responses.create(
    model=MODEL,
    input="Say Hello"
)

# Print the text response from the model to verify communication works
print(test_response.output_text)

Hello! How can I help you today?


**Define Tasks**

In [7]:
tasks = [
    {
        "task_id": "code_1",
        "task_type": "code_generation",
        "question": "Write a Python function to return the second largest unique number in a list.",
        "reference_points": ["function", "second largest", "unique", "list"]
    },
    {
        "task_id": "math_1",
        "task_type": "mathematical_reasoning",
        "question": "A product costs ₹800 after a 20% discount. What was the original price?",
        "reference_points": ["80%", "original", "1000"]
    },
    {
        "task_id": "sum_1",
        "task_type": "summarization",
        "question": "Summarize: AI adoption requires governance, scaling strategy, and alignment with business goals.",
        "reference_points": ["AI adoption", "governance", "scaling", "business"]
    }
]

print("Tasks loaded:", len(tasks))

Tasks loaded: 3


In [11]:
tasks[2]["question"]

'Summarize: AI adoption requires governance, scaling strategy, and alignment with business goals.'

**Create Prompt Function**

In [12]:
def build_prompt(task, strategy):
    """
    Constructs a prompt based on the chosen engineering strategy.

    Args:
        task (dict): A dictionary containing 'question' and 'task_type'
        strategy (str): One of 'zero_shot', 'cot' (Chain of Thought), or 'few_shot'
    """
    question = task["question"]

    # Strategy 1: Zero-shot (simple direct execution instructions without supplementary details or examples)
    if strategy == "zero_shot":
        return f"Answer this:\n{question}"

    # Strategy 2: Chain of Thought (explicitly instructs the model to approach the problem using step-by-step reasoning)
    elif strategy == "cot":
        return f"Solve step by step and give final answer:\n{question}"

    # Strategy 3: Few-shot (provides one exemplary input-output demonstration tailored to the specific category of the task)
    elif strategy == "few_shot":
        if task["task_type"] == "code_generation":
            return f"""
Example:
Task: Write a function to add two numbers
Answer:
def add(a,b):
    return a+b

Now solve:
{question}
"""
        elif task["task_type"] == "mathematical_reasoning":
            return f"""
Example:
Question: 50% of 200?
Answer: 100

Now solve:
{question}
"""
        elif task["task_type"] == "summarization":
            return f"""
Example:
Text: AI improves efficiency
Summary:
- Improves efficiency
- Helps decisions

Now summarize:
{question}
"""
    else:
        raise ValueError("Invalid strategy provided")

**Test Prompt**

In [13]:
print(build_prompt(tasks[0], "few_shot"))


Example:
Task: Write a function to add two numbers
Answer:
def add(a,b):
    return a+b

Now solve:
Write a Python function to return the second largest unique number in a list.



**Create Model Function**

In [14]:
def run_model(prompt):
    """
    Helper function to send a formatted prompt to the model and retrieve the output text.
    """
    # Create a chat completion/generation request using the set model and string input
    response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    # Return the clean text payload of the model's generated response
    return response.output_text

**Test Model Output**

In [18]:
prompt = build_prompt(tasks[0], "zero_shot")
prompt

'Answer this:\nWrite a Python function to return the second largest unique number in a list.'

In [19]:
prompt = build_prompt(tasks[0], "zero_shot")
output = run_model(prompt)

print("OUTPUT:\n", output)

OUTPUT:
 Certainly! Here's a Python function that returns the second largest unique number in a list:

```python
def second_largest_unique(nums):
    unique_nums = list(set(nums))  # Get unique numbers
    if len(unique_nums) < 2:
        return None  # Not enough unique numbers for a second largest
    unique_nums.sort(reverse=True)  # Sort in descending order
    return unique_nums[1]  # Return the second largest

# Example usage:
numbers = [4, 1, 2, 4, 5, 5, 3]
print(second_largest_unique(numbers))  # Output: 4
```

This function:

1. Converts the list to a set to get unique numbers.
2. Checks if there are at least two unique numbers.
3. Sorts the unique numbers in descending order.
4. Returns the second element, which is the second largest unique number.

If there aren't enough unique numbers, it returns `None`.


**Create Scoring Function**

In [20]:
def score_output(task, output):
    """
    Calculates a basic validation score based on keyword matching.

    Measures how many of the task's expected 'reference_points' keywords
    appear in the model's generated output (case-insensitive).
    """
    score = 0
    # Iterate through all necessary keywords/concepts defined in the task configuration
    for word in task["reference_points"]:
        # Increment score if the keyword is found in the lowercased target output
        if word.lower() in output.lower():
            score += 1
    return score

In [21]:
score_output(tasks[0],output)

4

**Run Full Experiment**

In [22]:
# Evaluate different prompt strategies over the task suite
strategies = ["zero_shot", "cot", "few_shot"]
results = []

# Nested iteration to test every strategy against every task
for task in tasks:
    for strategy in strategies:
        # 1. Build context/prompt dynamically using the current engineering strategy
        prompt = build_prompt(task, strategy)
        # 2. Query the LLM model to fetch the result
        output = run_model(prompt)
        # 3. Compute metric score based on keyword occurrence matching
        score = score_output(task, output)

        # Save the metadata and metric of this run
        results.append({
            "task": task["task_id"],
            "strategy": strategy,
            "score": score
        })

        # Log key results to check progress real-time
        print("="*50)
        print("Task:", task["task_id"])
        print("Strategy:", strategy)
        print("Score:", score)
        print("Output:\n", output)

Task: code_1
Strategy: zero_shot
Score: 4
Output:
 Certainly! Here's a Python function that returns the second largest unique number in a list:

```python
def second_largest_unique(nums):
    unique_nums = set(nums)  # Get unique numbers
    if len(unique_nums) < 2:
        return None  # Not enough unique numbers for second largest
    unique_nums.remove(max(unique_nums))  # Remove the largest
    return max(unique_nums)  # Now max is the second largest

# Example usage:
print(second_largest_unique([4, 1, 2, 2, 5, 5, 3]))  # Output: 4
```

This function:
- Converts the list to a set to get unique numbers.
- Checks if there are at least two unique numbers.
- Removes the largest number.
- Returns the next largest number, which is the second largest unique number.
Task: code_1
Strategy: cot
Score: 4
Output:
 Certainly! Here's how you can solve the problem step-by-step and then the complete Python function:

### Steps:
1. **Remove duplicates** from the list to consider only unique numbers

In [23]:
results

[{'task': 'code_1', 'strategy': 'zero_shot', 'score': 4},
 {'task': 'code_1', 'strategy': 'cot', 'score': 4},
 {'task': 'code_1', 'strategy': 'few_shot', 'score': 3},
 {'task': 'math_1', 'strategy': 'zero_shot', 'score': 2},
 {'task': 'math_1', 'strategy': 'cot', 'score': 3},
 {'task': 'math_1', 'strategy': 'few_shot', 'score': 3},
 {'task': 'sum_1', 'strategy': 'zero_shot', 'score': 4},
 {'task': 'sum_1', 'strategy': 'cot', 'score': 4},
 {'task': 'sum_1', 'strategy': 'few_shot', 'score': 3}]

**Convert to Table**

In [24]:
df = pd.DataFrame(results)
df

,task,strategy,score
0,code_1,zero_shot,4
1,code_1,cot,4
2,code_1,few_shot,3
3,math_1,zero_shot,2
4,math_1,cot,3
5,math_1,few_shot,3
6,sum_1,zero_shot,4
7,sum_1,cot,4
8,sum_1,few_shot,3


**Best Strategy**

In [25]:
df.loc[df.groupby("task")["score"].idxmax()]

,task,strategy,score
0,code_1,zero_shot,4
4,math_1,cot,3
6,sum_1,zero_shot,4


**Save Results**

In [26]:
df.to_csv("lab1_results.csv", index=False)
print("Saved successfully")

Saved successfully


**Lab 2: Implement a context engineering pipeline**

**Install required libraries**

In [27]:
!pip install -q openai pandas numpy tiktoken

**Load OpenAI API key from Colab Secrets**

In [28]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("API Key Loaded:", os.environ.get("OPENAI_API_KEY")[:8])

API Key Loaded: sk-proj-


**Create client and define models**

In [29]:
from openai import OpenAI
import pandas as pd
import numpy as np
import tiktoken
import math

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GEN_MODEL = "gpt-4.1-mini"
EMBED_MODEL = "text-embedding-3-small"

print("Client initialized successfully")

Client initialized successfully


In [36]:
response = client.embeddings.create(
  model=EMBED_MODEL,
  input="Amadou is listening to the class on a sunday",
)

print(response.data[0].embedding)

[0.006580352783203125, -0.00396728515625, -0.0250091552734375, -0.04241943359375, 0.00958251953125, 0.00079345703125, -0.0046844482421875, 0.015472412109375, 0.0178070068359375, -0.02923583984375, 0.016082763671875, -0.0670166015625, 0.000850677490234375, 0.02789306640625, 0.007572174072265625, -0.0018482208251953125, -0.02203369140625, 0.00887298583984375, 0.025970458984375, 0.0299072265625, 0.0277099609375, 0.0279541015625, 0.0123138427734375, 0.05645751953125, 0.01203155517578125, -0.027099609375, -0.005054473876953125, 0.042449951171875, 0.020233154296875, -0.002330780029296875, 0.065185546875, -0.04510498046875, -0.02764892578125, 0.0031585693359375, -0.038299560546875, -0.01934814453125, 0.0031833648681640625, -0.017303466796875, -0.0267791748046875, -0.061981201171875, -0.0086517333984375, -0.026885986328125, -0.006076812744140625, -0.04595947265625, 0.0214996337890625, -0.017181396484375, -0.0224761962890625, 0.016571044921875, 0.03375244140625, 0.0302734375, -0.004512786865234

In [37]:
len(response.data[0].embedding)

1536

**Test the API connection**

In [38]:
test_response = client.responses.create(
    model=GEN_MODEL,
    input="Reply with exactly: OpenAI connection works"
)

print(test_response.output_text)

OpenAI connection works


**Create a large synthetic knowledge base**

In [39]:
base_sections = [
    {
        "title": "Company Overview",
        "text": """
EduSphere is a fictional global learning platform that serves universities, enterprises, and independent learners.
Its strategy focuses on AI-assisted learning, multilingual education delivery, mobile-first access, and learner retention.
The company operates in India, Southeast Asia, Europe, and North America.
""".strip()
    },
    {
        "title": "Products",
        "text": """
EduSphere offers live classes, recorded programs, enterprise dashboards, AI study assistants, assessments, and certification workflows.
Its AI assistant supports summarization, revision plans, question answering, and practice generation.
Enterprise customers use dashboards to track completion, engagement, and skill progression.
""".strip()
    },
    {
        "title": "Governance",
        "text": """
The company enforces role-based access, audit logging, regional data controls, consent capture, model output monitoring, and review workflows.
Sensitive personal data is minimized in prompts where possible.
Human review is mandatory for high-stakes use cases such as compliance summaries, grading recommendations, and policy-sensitive outputs.
""".strip()
    },
    {
        "title": "Architecture",
        "text": """
The platform uses modular services for content, learner profiles, analytics, recommendations, and AI orchestration.
Long-form content is chunked, embedded, indexed, retrieved, ranked, and compressed before final model inference.
The architecture supports enterprise knowledge assistants and internal support copilots.
""".strip()
    },
    {
        "title": "Support Policy",
        "text": """
Critical enterprise incidents receive a response within one hour.
Billing issues are generally handled within one business day.
Regulatory or governance-related escalations are routed to a specialist review team.
""".strip()
    }
]

knowledge_base = []
for i in range(1, 501):
    section = base_sections[(i - 1) % len(base_sections)]
    knowledge_base.append(
        f"Document {i}\nTitle: {section['title']}\nContent: {section['text']}"
    )

full_kb_text = "\n\n".join(knowledge_base)

print("Number of documents:", len(knowledge_base))
print("Total characters:", len(full_kb_text))

Number of documents: 500
Total characters: 173090


In [41]:
knowledge_base

['Document 1\nTitle: Company Overview\nContent: EduSphere is a fictional global learning platform that serves universities, enterprises, and independent learners.\nIts strategy focuses on AI-assisted learning, multilingual education delivery, mobile-first access, and learner retention.\nThe company operates in India, Southeast Asia, Europe, and North America.',
 'Document 2\nTitle: Products\nContent: EduSphere offers live classes, recorded programs, enterprise dashboards, AI study assistants, assessments, and certification workflows.\nIts AI assistant supports summarization, revision plans, question answering, and practice generation.\nEnterprise customers use dashboards to track completion, engagement, and skill progression.',
 'Document 3\nTitle: Governance\nContent: The company enforces role-based access, audit logging, regional data controls, consent capture, model output monitoring, and review workflows.\nSensitive personal data is minimized in prompts where possible.\nHuman revie

**Estimate knowledge base token count**

In [42]:
encoding = tiktoken.get_encoding("cl100k_base")
kb_token_count = len(encoding.encode(full_kb_text))

print("Approximate KB token count:", kb_token_count)

Approximate KB token count: 31100


**Split the knowledge base into safe chunks**

In [43]:
def count_tokens(text):
    return len(encoding.encode(text))

def chunk_document(text, max_tokens=400):
    words = text.split()
    chunks = []
    current_words = []

    for word in words:
        trial = " ".join(current_words + [word])
        if count_tokens(trial) <= max_tokens:
            current_words.append(word)
        else:
            chunks.append(" ".join(current_words))
            current_words = [word]

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks

all_chunks = []

for doc_id, doc_text in enumerate(knowledge_base):
    doc_chunks = chunk_document(doc_text, max_tokens=400)
    for chunk_id, chunk_text in enumerate(doc_chunks):
        all_chunks.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "text": chunk_text,
            "tokens": count_tokens(chunk_text)
        })

print("Total chunks created:", len(all_chunks))
print("Sample chunk:\n")
print(all_chunks[0]["text"])
print("\nSample chunk tokens:", all_chunks[0]["tokens"])

Total chunks created: 500
Sample chunk:

Document 1 Title: Company Overview Content: EduSphere is a fictional global learning platform that serves universities, enterprises, and independent learners. Its strategy focuses on AI-assisted learning, multilingual education delivery, mobile-first access, and learner retention. The company operates in India, Southeast Asia, Europe, and North America.

Sample chunk tokens: 64


**Create a batch embeddings function**

In [44]:
def get_embeddings_batch(text_list, model=EMBED_MODEL):
    response = client.embeddings.create(
        model=model,
        input=text_list
    )
    return [item.embedding for item in response.data]

**Generate embeddings for all chunks**

In [45]:
batch_size = 50
chunk_texts = [item["text"] for item in all_chunks]

all_vectors = []

for start in range(0, len(chunk_texts), batch_size):
    batch = chunk_texts[start:start + batch_size]
    batch_vectors = get_embeddings_batch(batch)
    all_vectors.extend(batch_vectors)
    print(f"Processed batch {start // batch_size + 1} / {math.ceil(len(chunk_texts) / batch_size)}")

for i, vec in enumerate(all_vectors):
    all_chunks[i]["embedding"] = vec

print("Embeddings attached to all chunks:", len(all_chunks))

Processed batch 1 / 10
Processed batch 2 / 10
Processed batch 3 / 10
Processed batch 4 / 10
Processed batch 5 / 10
Processed batch 6 / 10
Processed batch 7 / 10
Processed batch 8 / 10
Processed batch 9 / 10
Processed batch 10 / 10
Embeddings attached to all chunks: 500


In [47]:
len(all_chunks[0]["embedding"])

1536

In [48]:
len(all_chunks[1]["embedding"])

1536

In [58]:
len(all_chunks[499]["embedding"])

1536

**Create cosine similarity and retrieval function**

In [49]:
def cosine_similarity(a, b):
    # Convert inputs to NumPy arrays for vectorized numerical operations
    a = np.array(a)
    b = np.array(b)
    # Calculate and return the cosine similarity: dot product divided by the product of L2 norms
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve_top_chunks(query, top_k=8):
    # Create a vector representation for the user query using the embedding model
    query_embedding = client.embeddings.create(
        model=EMBED_MODEL,
        input=query
    ).data[0].embedding

    scored_chunks = []

    # Loop through all text chunks in the knowledge base and compute similarity score
    for item in all_chunks:
        score = cosine_similarity(query_embedding, item["embedding"])
        scored_chunks.append({
            "doc_id": item["doc_id"],
            "chunk_id": item["chunk_id"],
            "text": item["text"],
            "tokens": item["tokens"],
            "similarity": float(score)
        })

    # Sort the scored chunks in descending order of similarity and return the top K matches
    scored_chunks = sorted(scored_chunks, key=lambda x: x["similarity"], reverse=True)
    return scored_chunks[:top_k]

**Test retrieval**

In [50]:
query = "How does EduSphere manage sensitive or high-stakes AI outputs?"

top_chunks = retrieve_top_chunks(query, top_k=5)

for i, item in enumerate(top_chunks, 1):
    print("=" * 80)
    print(f"Rank {i}")
    print("Similarity:", round(item["similarity"], 4))
    print("Tokens:", item["tokens"])
    print(item["text"][:800])

Rank 1
Similarity: 0.601
Tokens: 63
Document 382 Title: Products Content: EduSphere offers live classes, recorded programs, enterprise dashboards, AI study assistants, assessments, and certification workflows. Its AI assistant supports summarization, revision plans, question answering, and practice generation. Enterprise customers use dashboards to track completion, engagement, and skill progression.
Rank 2
Similarity: 0.6009
Tokens: 63
Document 62 Title: Products Content: EduSphere offers live classes, recorded programs, enterprise dashboards, AI study assistants, assessments, and certification workflows. Its AI assistant supports summarization, revision plans, question answering, and practice generation. Enterprise customers use dashboards to track completion, engagement, and skill progression.
Rank 3
Similarity: 0.6004
Tokens: 63
Document 392 Title: Products Content: EduSphere offers live classes, recorded programs, enterprise dashboards, AI study assistants, assessments, and certif

In [51]:
test_response = client.responses.create(
    model=GEN_MODEL,
    input= "Query" + query + "\n\nRelevant chunks:\n" + "\n".join([item["text"] for item in top_chunks])
)

print(test_response.output_text)

The provided documents about EduSphere describe its various offerings, including live classes, recorded programs, enterprise dashboards, AI study assistants, assessments, and certification workflows, with AI features like summarization, revision plans, question answering, and practice generation. However, none of the chunks explicitly detail how EduSphere manages sensitive or high-stakes AI outputs.

Therefore, based on the available information, there is no specific insight into EduSphere's approach or policies regarding the management of sensitive or high-stakes AI outputs.


**Add prioritization rules**

In [52]:
def prioritize_chunks(retrieved_chunks):
    # List of priority terms related to security, oversight, or compliance
    priority_terms = [
        "governance",
        "high-stakes",
        "sensitive",
        "audit",
        "review",
        "compliance"
    ]

    prioritized = []

    # Iterate through the retrieved matches and apply a similarity boost for keyword matches
    for item in retrieved_chunks:
        bonus = 0.0
        text_lower = item["text"].lower()

        # Add a scoring bonus of 0.02 for each distinct keyword found in the chunk
        for term in priority_terms:
            if term in text_lower:
                bonus += 0.02

        prioritized.append({
            **item,
            "priority_score": item["similarity"] + bonus
        })

    # Sort the prioritized chunks descending by their final similarity + bonus score
    prioritized = sorted(prioritized, key=lambda x: x["priority_score"], reverse=True)
    return prioritized

 **Compress each selected chunk**

In [53]:
def compress_chunk(query, chunk_text):
    prompt = f"""
You are compressing retrieved context for a downstream question-answering system.

User question:
{query}

Retrieved chunk:
{chunk_text}

Task:
Rewrite this chunk into exactly 3 concise bullet points.
Keep only details useful for answering the user question.
Do not add any new facts.
"""
    response = client.responses.create(
        model=GEN_MODEL,
        input=prompt
    )
    return response.output_text.strip()

**Build the context engineering pipeline**

In [54]:
def build_context(query, retrieve_k=8, final_budget_tokens=3000):
    # 1. Retrieve the top K chunks most visually/semantically relevant to the user query
    retrieved = retrieve_top_chunks(query, top_k=retrieve_k)
    # 2. Re-rank/prioritize chunks containing high-importance keywords
    prioritized = prioritize_chunks(retrieved)

    selected_context = []
    used_tokens = 0

    # 3. Iterate and compress retrieved content while staying under the token budget limit
    for item in prioritized:
        compressed = compress_chunk(query, item["text"])
        compressed_tokens = count_tokens(compressed)

        # Check if adding this chunk respects the maximum token budget
        if used_tokens + compressed_tokens <= final_budget_tokens:
            selected_context.append({
                "doc_id": item["doc_id"],
                "chunk_id": item["chunk_id"],
                "original_similarity": item["similarity"],
                "priority_score": item["priority_score"],
                "compressed_text": compressed,
                "compressed_tokens": compressed_tokens
            })
            used_tokens += compressed_tokens

    return selected_context, used_tokens

**Run the pipeline**

In [55]:
query = "How does EduSphere manage AI governance for sensitive and high-stakes use cases?"

selected_context, used_tokens = build_context(
    query=query,
    retrieve_k=8,
    final_budget_tokens=1500
)

print("Selected context items:", len(selected_context))
print("Used tokens:", used_tokens)

for i, item in enumerate(selected_context, 1):
    print("\n" + "=" * 80)
    print(f"Context Item {i}")
    print("Priority Score:", round(item["priority_score"], 4))
    print("Compressed Tokens:", item["compressed_tokens"])
    print(item["compressed_text"])

Selected context items: 8
Used tokens: 395

Context Item 1
Priority Score: 0.5871
Compressed Tokens: 55
- EduSphere is a global learning platform serving universities, enterprises, and independent learners.  
- Its strategy includes AI-assisted learning with a focus on multilingual education and mobile-first access.  
- The company operates across India, Southeast Asia, Europe, and North America.

Context Item 2
Priority Score: 0.5865
Compressed Tokens: 49
- EduSphere is a global learning platform serving universities, enterprises, and independent learners.  
- Its strategy includes AI-assisted learning and multilingual education delivery.  
- The company operates in India, Southeast Asia, Europe, and North America.

Context Item 3
Priority Score: 0.5863
Compressed Tokens: 48
- EduSphere is a global learning platform serving universities, enterprises, and independent learners.  
- It emphasizes AI-assisted learning as part of its core strategy.  
- Operates across India, Southeast Asia

**Generate the final answer from selected context**

In [56]:
def answer_with_context(query, selected_context):
    context_block = "\n\n".join(
        [f"Source {i+1}:\n{item['compressed_text']}" for i, item in enumerate(selected_context)]
    )

    prompt = f"""
You are an enterprise knowledge assistant.

Answer the user question using only the context below.
If the context is insufficient, say that clearly.

Context:
{context_block}

User question:
{query}

Instructions:
- Give a direct answer
- Use short paragraphs
- Do not invent facts
"""
    response = client.responses.create(
        model=GEN_MODEL,
        input=prompt
    )
    return response.output_text.strip()

final_answer = answer_with_context(query, selected_context)
print(final_answer)

The provided context does not include information on how EduSphere manages AI governance for sensitive and high-stakes use cases.


**Compare with a naive baseline**

In [59]:
naive_context = "\n\n".join([item["text"] for item in all_chunks[:8]])

naive_prompt = f"""
Answer the question using the context below.

Context:
{naive_context}

Question:
{query}
"""

naive_answer = client.responses.create(
    model=GEN_MODEL,
    input=naive_prompt
).output_text.strip()

print("NAIVE ANSWER:\n")
print(naive_answer)

print("\n" + "#" * 100 + "\n")

print("ENGINEERED CONTEXT ANSWER:\n")
print(final_answer)

NAIVE ANSWER:

EduSphere manages AI governance for sensitive and high-stakes use cases by implementing several key measures:

- **Role-based access and audit logging** to control and monitor who can interact with the AI systems and to keep detailed records of actions.
- **Regional data controls and consent capture** to ensure compliance with local regulations and that user permissions are properly obtained.
- **Model output monitoring and review workflows** to continuously oversee AI-generated results for quality and appropriateness.
- **Minimizing sensitive personal data in AI prompts** wherever possible to reduce privacy risks.
- **Mandatory human review for high-stakes scenarios**, such as compliance summaries, grading recommendations, and policy-sensitive outputs, ensuring that critical decisions are vetted by humans before finalization.

These governance practices collectively help EduSphere maintain control, compliance, and reliability in its AI-assisted learning offerings, parti

**Save outputs**

In [60]:
context_df = pd.DataFrame(selected_context)
context_df.to_csv("lab2_selected_context.csv", index=False)

with open("lab2_final_answer.txt", "w", encoding="utf-8") as f:
    f.write(final_answer)

print("Saved files:")
print("- lab2_selected_context.csv")
print("- lab2_final_answer.txt")

Saved files:
- lab2_selected_context.csv
- lab2_final_answer.txt
